## Langfuse Adapter Demo 

This notebook demonstrates how to use the generic **HallucinationEvaluator** with **EPRDetector** and **WEPRDetector** to automatically score LLM traces in Langfuse.

### Prerequisites

* Create a free LangFuse project at [cloud.langfuse.com](https://cloud.langfuse.com), then go to **Settings → API Keys**. 

* Obtain an access token from your Hugging Face Account.

* API keys must be set as environment variables.  

### Run a generation and send it to Langfuse

In [ ]:
import os
import time

from langfuse import get_client, observe
from langfuse.openai import OpenAI
from openai.types.chat import ChatCompletion

from artefactual.adapters.langfuse.evaluator import HallucinationEvaluator
from artefactual.scoring.hallucination_detector import EPRDetector, WEPRDetector

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])

@observe()
def run_generation() -> ChatCompletion:
    return client.chat.completions.create(
        model="Qwen/Qwen3-Coder-Next",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"},
        ],
        logprobs=True,
        top_logprobs=5,
    )


print("Generated message:", run_generation().choices[0].message.content)  # noqa: T201

langfuse = get_client()
langfuse.flush()

print("Waiting for Langfuse server to index the logporbs of the trace...")  # noqa: T201
time.sleep(3.0)

traces_to_evaluate = langfuse.api.trace.list(limit=1).data

Generated message: The capital of France is Paris.
Waiting for Langfuse server to index the logporbs of the trace...


### Score traces with EPR


In [ ]:
evaluator = HallucinationEvaluator(
    name="EPR",
    langfuse_client=langfuse,
    detector=EPRDetector(),
)

for trace in traces_to_evaluate:
    score = evaluator.score_trace(trace.id)
    print(f"EPR Scored Trace : {trace.id} → {score}")  # noqa: T201

langfuse.flush()

EPR Scored Trace : babbbea71b02e917e279f9daf9125afc → 0.2941811978816986


### Score traces with WEPR

In [ ]:
evaluator = HallucinationEvaluator(
    name="WEPR",
    langfuse_client=langfuse,
    detector=WEPRDetector(pretrained_model_name_or_path="weights_ministral.json"),
)



for trace in traces_to_evaluate:
    score = evaluator.score_trace(trace.id)
    print(f"WEPR Scored Trace : {trace.id} → {score}")  # noqa: T201

langfuse.flush()

WEPR Scored Trace : babbbea71b02e917e279f9daf9125afc → 0.06798840314149857
